# Notebook 05 - Benchmarks, Metricas y Monos

Este notebook usa los outputs de `Notebook_4_Ejecucion_y_Costes.ipynb` y construye:
- Benchmark estrategia vs SPY (mensual).
- Graficos comparativos.
- Simulacion Monte Carlo de "monos" (>= 25,000,000 carteras) con chunking.


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pyarrow.parquet as pq
import pyarrow.fs as fs

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)

SEED = 12345
rng = np.random.default_rng(SEED)
print(f"SEED fijo: {SEED}")


In [ ]:
# =========================
# Configuracion de paths
# =========================

OUTPUTS_DIR_CANDIDATES = [
    "notebooks/outputs",
    "outputs",
]

EQUITY_CSV_NAME = "backtest_equity_curve.csv"
TRADES_CSV_NAME = "trades.csv"
WEIGHTS_CSV_NAME = "weights_by_rebalance.csv"
SELECTION_CSV_NAME = "selected_top20_by_rebalance.csv"

PRICE_PATH_HINTS = [
    r"C:\Users\alons\Desktop\Practica 7\sp500_history.parquet",
    r"C:\Users\alons\Desktop\Pr?ctica 7\sp500_history.parquet",
    "data/raw/sp500_history.parquet",
    "notebooks/data/raw/sp500_history.parquet",
]

# Monte Carlo block-bootstrap momentum
N_MONOS = 25_000_000
MC_CHUNK_SIZE = 6
MC_BATCH_SIZE = 5_000_000
MONO_MONTHLY_COST = 0.0046  # 0.23% compra + 0.23% venta (turnover 100%)

# Bins fijos para histogramas
CAGR_BINS = np.linspace(-0.6, 1.2, 181)
FINAL_BINS = np.linspace(-1.0, 12.0, 261)

print("N_MONOS:", N_MONOS)
print("MC_CHUNK_SIZE:", MC_CHUNK_SIZE)
print("MC_BATCH_SIZE:", MC_BATCH_SIZE)
print("MONO_MONTHLY_COST:", MONO_MONTHLY_COST)

PRICE_SCAN_ROOTS = [".", "..", "notebooks", r"C:\Users\alons\Desktop"]


In [ ]:
# =========================
# Utilidades
# =========================

def resolve_existing_path(candidates):
    lfs = fs.LocalFileSystem()
    for p in candidates:
        try:
            info = lfs.get_file_info(p)
            if info.type == fs.FileType.File:
                return p
        except Exception:
            pass
    return None


def resolve_outputs_dir():
    lfs = fs.LocalFileSystem()
    for d in OUTPUTS_DIR_CANDIDATES:
        try:
            info = lfs.get_file_info(d)
            if info.type == fs.FileType.Directory:
                eq = f"{d}/{EQUITY_CSV_NAME}"
                tr = f"{d}/{TRADES_CSV_NAME}"
                if lfs.get_file_info(eq).type == fs.FileType.File and lfs.get_file_info(tr).type == fs.FileType.File:
                    return d
        except Exception:
            pass
    return None


def _canon(c):
    return str(c).strip().lower().replace(" ", "_").replace("-", "_")


def build_wide_close_from_long(df):
    x = df.copy()
    x.columns = [_canon(c) for c in x.columns]

    date_col = next((c for c in ["date", "datetime", "timestamp", "fecha"] if c in x.columns), None)
    ticker_col = next((c for c in ["symbol", "ticker", "asset", "activo"] if c in x.columns), None)

    close_col = None
    for c in ["adj_close", "adjclose", "close"]:
        if c in x.columns:
            close_col = c
            break

    if date_col is None or ticker_col is None or close_col is None:
        raise ValueError("No encuentro columnas long validas para construir close wide.")

    y = x[[date_col, ticker_col, close_col]].copy()
    y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
    y[ticker_col] = y[ticker_col].astype(str).str.upper()
    y[close_col] = pd.to_numeric(y[close_col], errors="coerce")
    y = y.dropna(subset=[date_col, ticker_col, close_col])

    wide = y.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc="last")
    wide = wide.sort_index().replace([np.inf, -np.inf], np.nan)
    wide = wide[~wide.index.duplicated(keep="last")]
    wide = wide.dropna(axis=1, how="all")
    return wide


def to_monthly_returns(px_wide):
    px_m = px_wide.resample("M").last()
    ret_m = px_m.pct_change(fill_method=None)
    return px_m, ret_m


def cagr_from_returns(r, periods_per_year=12):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    gross = (1.0 + r).prod()
    return gross ** (periods_per_year / len(r)) - 1.0


def max_drawdown_from_returns(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    return float(dd.min())


def beta_alpha_np(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    m = np.isfinite(y) & np.isfinite(x)
    y = y[m]
    x = x[m]
    if len(y) < 2:
        return np.nan, np.nan
    vx = np.var(x)
    if vx <= 1e-18:
        return np.nan, np.nan
    beta = np.cov(y, x, ddof=0)[0, 1] / vx
    alpha_m = y.mean() - beta * x.mean()
    alpha_ann = (1.0 + alpha_m) ** 12 - 1.0
    return float(beta), float(alpha_ann)


def metricas_mensuales(ret_m, ret_bench_m):
    r = pd.Series(ret_m).dropna()
    rb = pd.Series(ret_bench_m).dropna()
    idx = r.index.intersection(rb.index)
    r = r.reindex(idx).dropna()
    rb = rb.reindex(idx).dropna()

    if len(r) == 0:
        return {
            "CAGR": np.nan,
            "Vol anual": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "Max Drawdown": np.nan,
            "Beta": np.nan,
            "Alpha anual": np.nan,
        }

    mean_m = float(r.mean())
    std_m = float(r.std(ddof=0))
    vol_ann = std_m * np.sqrt(12.0)
    sharpe = (mean_m * 12.0 / vol_ann) if vol_ann > 1e-12 else np.nan

    downside = np.minimum(r.values, 0.0)
    downside_vol_ann = np.sqrt(np.mean(downside ** 2)) * np.sqrt(12.0)
    sortino = (mean_m * 12.0 / downside_vol_ann) if downside_vol_ann > 1e-12 else np.nan

    beta, alpha_ann = beta_alpha_np(r.values, rb.values)

    return {
        "CAGR": cagr_from_returns(r, periods_per_year=12),
        "Vol anual": vol_ann,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": max_drawdown_from_returns(r),
        "Beta": beta,
        "Alpha anual": alpha_ann,
    }


def resolve_price_path(hints, scan_roots=None, filename="sp500_history.parquet"):
    p0 = resolve_existing_path(hints)
    if p0 is not None:
        return p0

    lfs = fs.LocalFileSystem()
    roots = scan_roots if scan_roots is not None else [".", "..", "notebooks", r"C:\Users\alons\Desktop"]
    seen = set()
    for root in roots:
        try:
            infos = lfs.get_file_info(fs.FileSelector(root, recursive=True))
        except Exception:
            continue
        for info in infos:
            if info.type != fs.FileType.File:
                continue
            pp = str(info.path)
            if pp.lower().endswith(filename.lower()) and pp not in seen:
                seen.add(pp)
                return pp
    return None



def monte_carlo_momentum(monthly_returns, n_monos=25_000_000, chunk_size=6, fee_rate=0.0023, batch_size=5_000_000):
    """
    Simula Monte Carlo por Block Bootstrap para una serie mensual (momentum con regla de liquidez).

    Restricciones implementadas:
    - Solo NumPy dentro de la simulacion.
    - Unico bucle for: por lotes de memoria.
    - Sin guardar trayectorias completas: solo retorno acumulado final por simulacion.

    Parametros
    ----------
    monthly_returns : array-like
        Serie 1D de retornos mensuales historicos.
    n_monos : int
        Numero total de simulaciones.
    chunk_size : int
        Longitud de bloque bootstrap (meses).
    fee_rate : float
        Coste mensual fijo a restar al retorno de cada mes simulado.
    batch_size : int
        Numero de simulaciones por lote.

    Returns
    -------
    np.ndarray (float32)
        Retorno acumulado final de cada simulacion.
    """
    # 1) Preparacion y validaciones basicas.
    r = np.asarray(monthly_returns, dtype=np.float32).reshape(-1)
    r = r[np.isfinite(r)]

    if r.size < 2:
        raise ValueError("monthly_returns debe tener al menos 2 observaciones validas.")

    n_monos = int(n_monos)
    chunk_size = int(chunk_size)
    batch_size = int(batch_size)

    if n_monos <= 0:
        raise ValueError("n_monos debe ser > 0.")
    if chunk_size <= 0:
        raise ValueError("chunk_size debe ser > 0.")
    if batch_size <= 0:
        raise ValueError("batch_size debe ser > 0.")

    n_months = int(r.size)
    if chunk_size > n_months:
        chunk_size = n_months

    # 2) Preasignacion del array final (solo retorno acumulado por simulacion).
    final_returns = np.empty(n_monos, dtype=np.float32)

    # 3) RNG fuera del bucle de lotes (como pediste).
    rng = np.random.default_rng()

    # Numero de bloques necesarios para cubrir n_months por simulacion.
    n_blocks = (n_months + chunk_size - 1) // chunk_size

    # Inicios validos de bloque (inclusive).
    max_start = n_months - chunk_size + 1

    # 4) Unico bucle permitido: por lotes de memoria.
    for i0 in range(0, n_monos, batch_size):
        b = min(batch_size, n_monos - i0)

        # 4.1) Indices de inicio de bloque aleatorios para todo el lote.
        starts = rng.integers(0, max_start, size=(b, n_blocks), endpoint=False, dtype=np.int32)

        # 4.2) Broadcasting para expandir bloques y construir matriz 2D (b x n_months).
        offsets = np.arange(chunk_size, dtype=np.int32)[None, :, None]
        # shape: (b, chunk_size, n_blocks)
        idx3 = starts[:, None, :] + offsets
        # reshape a 2D y recorte exacto a n_months
        idx2 = idx3.reshape(b, n_blocks * chunk_size)[:, :n_months]

        # 4.3) Mapeo indices -> retornos reales.
        sim = r[idx2]  # float32, shape (b, n_months)

        # 4.4) Regla momentum con np.where (vectorizada):
        #      si el mes previo fue negativo, el mes actual pasa a 0.0 (liquidez).
        sim_adj = sim.copy()
        neg_prev = sim[:, :-1] < np.float32(0.0)
        sim_adj[:, 1:] = np.where(neg_prev, np.float32(0.0), sim_adj[:, 1:])

        # 4.5) Coste mensual.
        sim_adj = sim_adj - np.float32(fee_rate)

        # Proteccion numerica para log1p (evita valores <= -1).
        sim_adj = np.maximum(sim_adj, np.float32(-0.9999))

        # 4.6) Retorno acumulado final por simulacion:
        #      exp(sum(log1p(r_t))) - 1
        growth_log = np.sum(np.log1p(sim_adj.astype(np.float64)), axis=1, dtype=np.float64)
        final = np.exp(growth_log) - 1.0

        # 4.7) Guardado en el array preasignado.
        final_returns[i0:i0 + b] = final.astype(np.float32)

    return final_returns


In [ ]:
# =========================
# Carga estrategia (outputs NB4)
# =========================

outputs_dir = resolve_outputs_dir()
if outputs_dir is None:
    raise FileNotFoundError("No encuentro outputs con backtest_equity_curve.csv + trades.csv")

print("Outputs dir:", outputs_dir)

equity_path = f"{outputs_dir}/{EQUITY_CSV_NAME}"
trades_path = f"{outputs_dir}/{TRADES_CSV_NAME}"
weights_path = f"{outputs_dir}/{WEIGHTS_CSV_NAME}"
selection_path = f"{outputs_dir}/{SELECTION_CSV_NAME}"

equity_df = pd.read_csv(equity_path)
trades_df = pd.read_csv(trades_path)
weights_df = pd.read_csv(weights_path) if os.path.exists(weights_path) else pd.DataFrame()
selection_df = pd.read_csv(selection_path) if os.path.exists(selection_path) else pd.DataFrame()

equity_df["date"] = pd.to_datetime(equity_df["date"], errors="coerce")
equity_df = equity_df.dropna(subset=["date"]).sort_values("date")

equity_s = equity_df.set_index("date")["equity"].astype(float)
equity_m = equity_s.resample("M").last().dropna()
ret_strat_m = equity_m.pct_change(fill_method=None).dropna()

if "fee" in trades_df.columns:
    trades_df["fee"] = pd.to_numeric(trades_df["fee"], errors="coerce").fillna(0.0)
    total_fees_real = float(trades_df["fee"].sum())
else:
    total_fees_real = np.nan

print("Rango estrategia mensual:", ret_strat_m.index.min(), "->", ret_strat_m.index.max())
print("Meses estrategia:", len(ret_strat_m))
print("Total comisiones reales (trade_log):", round(total_fees_real, 2))


In [ ]:
# =========================
# Carga universo + SPY
# =========================

price_path = resolve_price_path(PRICE_PATH_HINTS, scan_roots=PRICE_SCAN_ROOTS)
if price_path is None:
    raise FileNotFoundError("No se encontro parquet de precios (hints + scan). Revisa PRICE_PATH_HINTS/PRICE_SCAN_ROOTS")

print("Precio fuente:", price_path)

t0 = time.perf_counter()
use_cols = ["date", "symbol", "adj_close", "close", "in_sp500"]
try:
    tb = pq.read_table(price_path, columns=use_cols)
except Exception:
    tb = pq.read_table(price_path)
raw_prices = tb.to_pandas()
print(f"Carga parquet: {time.perf_counter()-t0:.2f}s | rows={len(raw_prices):,}")

close_w = build_wide_close_from_long(raw_prices)
close_m, ret_univ_m = to_monthly_returns(close_w)

# SPY preferente desde panel local, fallback yfinance
if "SPY" in close_w.columns:
    spy_px = close_w["SPY"].dropna().copy()
    spy_source = "local_parquet"
else:
    s = ret_strat_m.index.min().strftime("%Y-%m-%d")
    e = (ret_strat_m.index.max() + pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    spy = yf.download("SPY", start=s, end=e, progress=False, auto_adjust=False)
    if isinstance(spy.columns, pd.MultiIndex):
        if ("Adj Close", "SPY") in spy.columns:
            spy_px = spy[("Adj Close", "SPY")]
        elif ("Close", "SPY") in spy.columns:
            spy_px = spy[("Close", "SPY")]
        else:
            spy_px = spy.xs("Close", axis=1, level=0).iloc[:, 0]
    else:
        spy_px = spy["Adj Close"] if "Adj Close" in spy.columns else spy["Close"]
    spy_px.index = pd.to_datetime(spy_px.index)
    spy_source = "yfinance"

spy_m = spy_px.resample("M").last().dropna()
ret_spy_m = spy_m.pct_change(fill_method=None).dropna()

# Alineacion base benchmark mensual
idx_base = ret_strat_m.index.intersection(ret_spy_m.index)
ret_strat_b = ret_strat_m.reindex(idx_base).dropna()
ret_spy_b = ret_spy_m.reindex(idx_base).dropna()
idx_base = ret_strat_b.index.intersection(ret_spy_b.index)
ret_strat_b = ret_strat_b.reindex(idx_base)
ret_spy_b = ret_spy_b.reindex(idx_base)

print("Fuente SPY:", spy_source)
print("Meses benchmark comunes:", len(idx_base))
print("Universo mensual shape (T x N):", ret_univ_m.shape)


In [ ]:
# =========================
# 1) Tabla de metricas mensuales
# =========================

m_strategy = metricas_mensuales(ret_strat_b, ret_spy_b)
m_spy = metricas_mensuales(ret_spy_b, ret_spy_b)

# Por definicion, benchmark contra si mismo
m_spy["Beta"] = 1.0
m_spy["Alpha anual"] = 0.0

metrics_df = pd.DataFrame([m_strategy, m_spy], index=["Strategy", "SPY"])

fmt_pct_cols = ["CAGR", "Vol anual", "Sortino", "Max Drawdown", "Alpha anual"]
show_df = metrics_df.copy()
for c in ["CAGR", "Vol anual", "Max Drawdown", "Alpha anual"]:
    show_df[c] = show_df[c].map(lambda x: f"{x:.2%}" if pd.notna(x) else "NaN")
for c in ["Sharpe", "Sortino", "Beta"]:
    show_df[c] = show_df[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "NaN")

print("Metricas mensuales (Strategy vs SPY)")
display(show_df)

strategy_sharpe_ref = float(metrics_df.loc["Strategy", "Sharpe"])
strategy_cagr_ref = float(metrics_df.loc["Strategy", "CAGR"])
print("Sharpe referencia estrategia:", round(strategy_sharpe_ref, 6))
print("CAGR referencia estrategia:", round(strategy_cagr_ref, 6))


In [ ]:
# =========================
# 2) Graficos obligatorios
# =========================

# 2.1 Rentabilidad acumulada (%) estrategia vs SPY
cum_strategy = (1.0 + ret_strat_b).cumprod() - 1.0
cum_spy = (1.0 + ret_spy_b).cumprod() - 1.0

plt.figure(figsize=(12, 5))
plt.plot(cum_strategy.index, cum_strategy.values * 100, label="Strategy")
plt.plot(cum_spy.index, cum_spy.values * 100, label="SPY")
plt.title("Rentabilidad acumulada (%) - Strategy vs SPY")
plt.ylabel("%")
plt.legend()
plt.tight_layout()
plt.show()

# 2.2 Histograma retornos mensuales
plt.figure(figsize=(12, 5))
plt.hist(ret_strat_b.values * 100, bins=40, alpha=0.6, label="Strategy")
plt.hist(ret_spy_b.values * 100, bins=40, alpha=0.6, label="SPY")
plt.title("Histograma retornos mensuales (%)")
plt.xlabel("Retorno mensual (%)")
plt.legend()
plt.tight_layout()
plt.show()

# 2.3 Recurrencia: scatter anuales y trimestrales
ann_strat = (1.0 + ret_strat_b).resample("Y").prod() - 1.0
ann_spy = (1.0 + ret_spy_b).resample("Y").prod() - 1.0
ann_idx = ann_strat.index.intersection(ann_spy.index)
ann_strat = ann_strat.reindex(ann_idx)
ann_spy = ann_spy.reindex(ann_idx)

q_strat = (1.0 + ret_strat_b).resample("Q").prod() - 1.0
q_spy = (1.0 + ret_spy_b).resample("Q").prod() - 1.0
q_idx = q_strat.index.intersection(q_spy.index)
q_strat = q_strat.reindex(q_idx)
q_spy = q_spy.reindex(q_idx)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(ann_spy.values * 100, ann_strat.values * 100, alpha=0.8)
mn = min(ann_spy.min(), ann_strat.min()) * 100
mx = max(ann_spy.max(), ann_strat.max()) * 100
axes[0].plot([mn, mx], [mn, mx], "r--", linewidth=1)
axes[0].set_title("Recurrencia anual: Strategy vs SPY")
axes[0].set_xlabel("SPY anual (%)")
axes[0].set_ylabel("Strategy anual (%)")

axes[1].scatter(q_spy.values * 100, q_strat.values * 100, alpha=0.5)
mnq = min(q_spy.min(), q_strat.min()) * 100
mxq = max(q_spy.max(), q_strat.max()) * 100
axes[1].plot([mnq, mxq], [mnq, mxq], "r--", linewidth=1)
axes[1].set_title("Recurrencia trimestral: Strategy vs SPY")
axes[1].set_xlabel("SPY trimestral (%)")
axes[1].set_ylabel("Strategy trimestral (%)")

plt.tight_layout()
plt.show()


## 3) Monte Carlo Block Bootstrap (vectorizado)

Implementacion con la funcion `monte_carlo_momentum(...)`:
- Unico bucle `for`: por lotes de memoria (`batch_size`).
- Sin bucles por simulacion ni por mes en la celda de ejecucion.
- Coste mensual fijo de monos: `0.0046`.
- Se guarda solo retorno final por simulacion.


In [ ]:
# =========================
# Input Monte Carlo (serie mensual estrategia)
# =========================

idx_mc = ret_strat_b.index.intersection(ret_spy_b.index)
strat_mc = ret_strat_b.reindex(idx_mc).dropna()
spy_mc = ret_spy_b.reindex(idx_mc).dropna()
idx_mc = strat_mc.index.intersection(spy_mc.index)
strat_mc = strat_mc.reindex(idx_mc)
spy_mc = spy_mc.reindex(idx_mc)

if len(strat_mc) < 24:
    print("WARNING: muy pocos meses para inferencia robusta.")

strategy_metrics_mc = metricas_mensuales(strat_mc, spy_mc)
strategy_sharpe_mc = float(strategy_metrics_mc["Sharpe"])
strategy_cagr_mc = float(strategy_metrics_mc["CAGR"])
strategy_total_mc = float((1.0 + strat_mc).prod() - 1.0)

monthly_returns_mc = strat_mc.to_numpy(dtype=np.float32)

print("Meses usados para MC:", monthly_returns_mc.size)
print("Sharpe estrategia (ventana MC):", round(strategy_sharpe_mc, 6))
print("CAGR estrategia (ventana MC):", round(strategy_cagr_mc, 6))
print("Retorno total estrategia (ventana MC):", round(strategy_total_mc, 6))


In [ ]:
# =========================
# Simulacion Monte Carlo (sin for por meses)
# =========================

start = time.perf_counter()

mono_final = monte_carlo_momentum(
    monthly_returns=monthly_returns_mc,
    n_monos=N_MONOS,
    chunk_size=MC_CHUNK_SIZE,
    fee_rate=MONO_MONTHLY_COST,
    batch_size=MC_BATCH_SIZE,
)

elapsed_total = time.perf_counter() - start
throughput_total = N_MONOS / elapsed_total if elapsed_total > 0 else np.nan

# CAGR por mono a partir del retorno acumulado final
T_mc = int(monthly_returns_mc.size)
base = np.maximum(1.0 + mono_final.astype(np.float64), 1e-12)
mono_cagr = np.power(base, 12.0 / T_mc) - 1.0

finite_cg = np.isfinite(mono_cagr)
finite_final = np.isfinite(mono_final)

count_ge_cagr = int(np.sum(mono_cagr[finite_cg] >= strategy_cagr_mc))
count_ge_final = int(np.sum(mono_final[finite_final] >= strategy_total_mc))

p_value_cagr = count_ge_cagr / max(1, int(np.sum(finite_cg)))
p_value_final = count_ge_final / max(1, int(np.sum(finite_final)))

hist_cagr = np.histogram(mono_cagr[finite_cg], bins=CAGR_BINS)[0]
hist_final = np.histogram(mono_final[finite_final], bins=FINAL_BINS)[0]

print("Monte Carlo terminado")
print(f"Tiempo total: {elapsed_total:,.2f}s")
print(f"Throughput promedio: {throughput_total:,.0f} monos/s")
print("p-value (CAGR mono >= CAGR estrategia):", p_value_cagr)
print("p-value (Retorno final mono >= retorno final estrategia):", p_value_final)


In [ ]:
# =========================
# Plots finales Monte Carlo
# =========================

cg_centers = 0.5 * (CAGR_BINS[:-1] + CAGR_BINS[1:])
rf_centers = 0.5 * (FINAL_BINS[:-1] + FINAL_BINS[1:])

cdf_cg = np.cumsum(hist_cagr) / max(1, hist_cagr.sum())
cdf_rf = np.cumsum(hist_final) / max(1, hist_final.sum())

pos_cg = np.searchsorted(cg_centers, strategy_cagr_mc, side="right") - 1
pos_cg = int(np.clip(pos_cg, 0, len(cdf_cg) - 1))
percentil_cg = float(cdf_cg[pos_cg])

pos_rf = np.searchsorted(rf_centers, strategy_total_mc, side="right") - 1
pos_rf = int(np.clip(pos_rf, 0, len(cdf_rf) - 1))
percentil_rf = float(cdf_rf[pos_rf])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cg_centers * 100.0, hist_cagr, width=(CAGR_BINS[1] - CAGR_BINS[0]) * 100.0, alpha=0.75)
axes[0].axvline(strategy_cagr_mc * 100.0, color="red", linestyle="--", linewidth=2, label="CAGR estrategia")
axes[0].set_title("Distribucion CAGR - Block Bootstrap")
axes[0].set_xlabel("CAGR (%)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

axes[1].bar(rf_centers * 100.0, hist_final, width=(FINAL_BINS[1] - FINAL_BINS[0]) * 100.0, alpha=0.75)
axes[1].axvline(strategy_total_mc * 100.0, color="red", linestyle="--", linewidth=2, label="Retorno total estrategia")
axes[1].set_title("Distribucion Retorno Final - Block Bootstrap")
axes[1].set_xlabel("Retorno final (%)")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Sharpe estrategia (referencia):", round(strategy_sharpe_mc, 6))
print("CAGR estrategia:", round(strategy_cagr_mc, 6))
print("p-value CAGR:", p_value_cagr)
print("Percentil CAGR estrategia (aprox):", round(percentil_cg * 100.0, 2), "%")
print("Retorno final estrategia:", round(strategy_total_mc, 6))
print("p-value retorno final:", p_value_final)
print("Percentil retorno final estrategia (aprox):", round(percentil_rf * 100.0, 2), "%")


## 4) Cierre critico

### Lectura rapida
- Si el p-value es alto, la estrategia no destaca frente a monos aleatorios bajo esta simplificacion.
- Si el p-value es bajo, hay evidencia de edge estadistico en la metrica evaluada.

### Limitaciones importantes
- Los monos usan coste simplificado fijo (`0.46%` mensual por rebalanceo total), sin minimo por orden.
- En ejecucion real hay minimo de comision por trade y slippage impl?cito, por lo que el coste efectivo puede ser distinto.
- Hay riesgo de sesgos por calidad de datos (precios ajustados/no ajustados, corporate actions, supervivencia y cobertura historica).
- Parametrizacion y filtros pueden inducir overfitting; conviene validacion fuera de muestra y sensibilidad de parametros.
- El rebalanceo (mensual/trimestral/semestral) impacta fuertemente turnover, fees y resultado final.

### Comisiones reales del backtest
Se reportan con `sum(trade_log.fee)` desde `trades.csv`.


In [ ]:
print("Comisiones reales pagadas (sum fee trades.csv):", round(total_fees_real, 2))
print("Configuracion monos -> coste mensual fijo:", MONO_MONTHLY_COST)
print("Nota: este coste de monos es una simplificacion respecto a fee minimo real por orden.")
